# FLUX GridToWave Training — Contrastive Learning

Train GridToWave to produce discriminative embeddings using random ARC-style grids.

**Approach:**
- Generate random grids with colored objects
- Train with contrastive loss: different grids → different waves
- Modified grids → somewhat similar waves
- Same grids → identical waves

No external datasets needed — self-supervised training on synthetic grids.

---
## Cell 1: Install Dependencies

In [ ]:
%%time
# Install dependencies
!pip install -q torch matplotlib tqdm huggingface_hub

# Clone FLUX repo
import os
from pathlib import Path

ROOT = Path('/kaggle/working/FLUX')
if not ROOT.exists():
    !git clone https://github.com/Unseengap/FLUX.git {ROOT}
os.chdir(ROOT)

print('✓ Dependencies installed')
print(f'Working directory: {os.getcwd()}')

## Cell 2: Setup Paths & Device

In [ ]:
import sys
import torch
from pathlib import Path

ROOT = Path('/kaggle/working/FLUX')
PHASES_DIR = ROOT / 'phases'

# Add paths
for p in [ROOT, PHASES_DIR / 'phase9_arc', PHASES_DIR / 'phase8_8']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 3: Random Grid Generators

Generate ARC-style training data:
- Random colored grids with objects
- Modified versions (for similarity targets)
- Various patterns and densities

In [ ]:
import random
import torch

def generate_random_grid(size=10, num_objects=4, max_object_size=3):
    """Generate random ARC-style grid with colored objects."""
    grid = [[0] * size for _ in range(size)]
    
    for _ in range(num_objects):
        # Random object type
        obj_type = random.choice(['point', 'line', 'rect', 'L'])
        color = random.randint(1, 9)
        r, c = random.randint(0, size-1), random.randint(0, size-1)
        
        if obj_type == 'point':
            grid[r][c] = color
        elif obj_type == 'line':
            length = random.randint(2, max_object_size)
            horizontal = random.choice([True, False])
            for i in range(length):
                if horizontal and c + i < size:
                    grid[r][c + i] = color
                elif not horizontal and r + i < size:
                    grid[r + i][c] = color
        elif obj_type == 'rect':
            h = random.randint(2, max_object_size)
            w = random.randint(2, max_object_size)
            for dr in range(h):
                for dc in range(w):
                    if r + dr < size and c + dc < size:
                        grid[r + dr][c + dc] = color
        elif obj_type == 'L':
            leg = random.randint(2, max_object_size)
            for i in range(leg):
                if r + i < size:
                    grid[r + i][c] = color
            for j in range(1, leg):
                if r + leg - 1 < size and c + j < size:
                    grid[r + leg - 1][c + j] = color
    
    return grid

def modify_grid(grid, num_changes=3):
    """Create modified version of grid."""
    new_grid = [row[:] for row in grid]
    size = len(grid)
    for _ in range(num_changes):
        r, c = random.randint(0, size-1), random.randint(0, size-1)
        new_grid[r][c] = random.randint(0, 9)
    return new_grid

def apply_transformation(grid, transform='none'):
    """Apply ARC-like transformation."""
    size = len(grid)
    
    if transform == 'rotate90':
        return [[grid[size-1-c][r] for c in range(size)] for r in range(size)]
    elif transform == 'flipH':
        return [row[::-1] for row in grid]
    elif transform == 'flipV':
        return grid[::-1]
    elif transform == 'recolor':
        # Swap two colors
        c1, c2 = random.sample(range(1, 10), 2)
        return [[c2 if v == c1 else c1 if v == c2 else v for v in row] for row in grid]
    else:
        return grid

# Test generators
print('Testing grid generators...')
g1 = generate_random_grid(10, 4)
g2 = modify_grid(g1, 3)
g3 = apply_transformation(g1, 'rotate90')

print(f'Original grid preview:')
for row in g1[:5]:
    print(' ', row)
print()
print('✓ Grid generators ready')

## Cell 4: Training Batch Generator

In [ ]:
def generate_training_batch(batch_size=16, grid_size=10):
    """
    Generate a batch of training triplets:
    - anchor: random grid
    - different: completely different random grid  
    - modified: slightly modified anchor
    - transformed: same anchor with ARC transformation
    """
    batch = {
        'anchors': [],
        'differents': [],
        'modifieds': [],
        'transformeds': [],
    }
    
    transforms = ['rotate90', 'flipH', 'flipV', 'recolor']
    
    for _ in range(batch_size):
        anchor = generate_random_grid(grid_size, random.randint(2, 6))
        different = generate_random_grid(grid_size, random.randint(2, 6))
        modified = modify_grid(anchor, num_changes=random.randint(1, 4))
        transformed = apply_transformation(anchor, random.choice(transforms))
        
        batch['anchors'].append(anchor)
        batch['differents'].append(different)
        batch['modifieds'].append(modified)
        batch['transformeds'].append(transformed)
    
    return batch

# Test batch generation
batch = generate_training_batch(4)
print(f'Batch keys: {list(batch.keys())}')
print(f'Samples per key: {len(batch["anchors"])}')
print('✓ Batch generator ready')

## Cell 5: Training Configuration

In [ ]:
# Training hyperparameters
TRAIN_STEPS = 2000
BATCH_SIZE = 16
GRID_SIZE = 10
LR = 1e-4
GRAD_CLIP = 0.5
LOG_EVERY = 100

# Similarity targets for contrastive loss
SIM_DIFFERENT = 0.3   # Different grids → low similarity
SIM_MODIFIED = 0.7    # Modified grids → medium similarity  
SIM_TRANSFORMED = 0.9 # Transformed grids → high similarity

print('Training Configuration:')
print(f'  Steps: {TRAIN_STEPS}')
print(f'  Batch size: {BATCH_SIZE}')
print(f'  Grid size: {GRID_SIZE}x{GRID_SIZE}')
print(f'  Learning rate: {LR}')
print(f'  Gradient clipping: {GRAD_CLIP}')
print()
print('Similarity targets:')
print(f'  Different grids → {SIM_DIFFERENT}')
print(f'  Modified grids → {SIM_MODIFIED}')
print(f'  Transformed grids → {SIM_TRANSFORMED}')

## Cell 6: Initialize GridToWave

In [ ]:
from grid_adapters import GridToWave

WAVE_DIM = 432

# Fresh initialization
encoder = GridToWave(
    wave_dim=WAVE_DIM,
    num_colors=10,
    max_size=30,
    device=device,
)
encoder.to(device)

# Count parameters
params = sum(p.numel() for p in encoder.parameters())
print(f'GridToWave parameters: {params:,}')
print(f'Device: {device}')
print('✓ Model initialized')

## Cell 7: Contrastive Training Loop

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

# Put in training mode
encoder.train()

optimizer = torch.optim.Adam(encoder.parameters(), lr=LR)

def compute_contrastive_loss(encoder, batch):
    """
    Compute contrastive loss for a batch.
    
    Targets:
    - anchor vs different → low similarity (SIM_DIFFERENT)
    - anchor vs modified → medium similarity (SIM_MODIFIED)
    - anchor vs transformed → high similarity (SIM_TRANSFORMED)
    """
    total_loss = torch.tensor(0.0, device=device, requires_grad=True)
    valid = 0
    
    for i in range(len(batch['anchors'])):
        anchor = batch['anchors'][i]
        different = batch['differents'][i]
        modified = batch['modifieds'][i]
        transformed = batch['transformeds'][i]
        
        # Encode all
        w_anchor = encoder.encode(anchor, mode='holistic')
        w_different = encoder.encode(different, mode='holistic')
        w_modified = encoder.encode(modified, mode='holistic')
        w_transformed = encoder.encode(transformed, mode='holistic')
        
        # Skip if any NaN
        if (torch.isnan(w_anchor).any() or torch.isnan(w_different).any() or
            torch.isnan(w_modified).any() or torch.isnan(w_transformed).any()):
            continue
        
        # Cosine similarities
        sim_diff = F.cosine_similarity(w_anchor.unsqueeze(0), w_different.unsqueeze(0)).squeeze()
        sim_mod = F.cosine_similarity(w_anchor.unsqueeze(0), w_modified.unsqueeze(0)).squeeze()
        sim_trans = F.cosine_similarity(w_anchor.unsqueeze(0), w_transformed.unsqueeze(0)).squeeze()
        
        # MSE to target similarities
        loss_diff = (sim_diff - SIM_DIFFERENT).pow(2)
        loss_mod = (sim_mod - SIM_MODIFIED).pow(2)
        loss_trans = (sim_trans - SIM_TRANSFORMED).pow(2)
        
        # Magnitude regularization (prevent collapse)
        loss_mag = torch.relu(2.0 - w_anchor.norm())
        
        sample_loss = loss_diff + loss_mod + loss_trans + 0.01 * loss_mag
        total_loss = total_loss + sample_loss
        valid += 1
    
    if valid == 0:
        return torch.tensor(0.0, device=device, requires_grad=True), 0
    
    return total_loss / valid, valid

print('Training loop defined')
print('Run next cell to start training')

## Cell 8: Train Model

In [ ]:
%%time
print('Starting contrastive training...')
print('=' * 50)

history = {
    'loss': [],
    'sim_diff': [],
    'sim_mod': [],
    'sim_trans': [],
}

pbar = tqdm(range(TRAIN_STEPS), desc='Training')

for step in pbar:
    optimizer.zero_grad()
    
    # Generate fresh batch
    batch = generate_training_batch(BATCH_SIZE, GRID_SIZE)
    
    # Compute loss
    loss, valid = compute_contrastive_loss(encoder, batch)
    
    if valid == 0:
        continue
    
    # Backward
    loss.backward()
    
    # Gradient clipping
    torch.nn.utils.clip_grad_norm_(encoder.parameters(), GRAD_CLIP)
    
    # Step
    optimizer.step()
    
    # Track loss
    history['loss'].append(loss.item())
    
    # Log every 100 steps
    if (step + 1) % 100 == 0:
        avg_loss = sum(history['loss'][-100:]) / 100
        pbar.set_postfix({'loss': f'{avg_loss:.4f}'})
        
        # Sample similarity check
        with torch.no_grad():
            b = generate_training_batch(1, GRID_SIZE)
            w_a = encoder.encode(b['anchors'][0], mode='holistic')
            w_d = encoder.encode(b['differents'][0], mode='holistic')
            w_m = encoder.encode(b['modifieds'][0], mode='holistic')
            w_t = encoder.encode(b['transformeds'][0], mode='holistic')
            
            sim_d = F.cosine_similarity(w_a.unsqueeze(0), w_d.unsqueeze(0)).item()
            sim_m = F.cosine_similarity(w_a.unsqueeze(0), w_m.unsqueeze(0)).item()
            sim_t = F.cosine_similarity(w_a.unsqueeze(0), w_t.unsqueeze(0)).item()
            
            history['sim_diff'].append(sim_d)
            history['sim_mod'].append(sim_m)
            history['sim_trans'].append(sim_t)

print('\n✓ Training complete!')
print(f'Final loss: {sum(history["loss"][-10:])/10:.4f}')
print(f'Similarities: diff={sim_d:.2f} (→{SIM_DIFFERENT}), mod={sim_m:.2f} (→{SIM_MODIFIED}), trans={sim_t:.2f} (→{SIM_TRANSFORMED})')

## Cell 9: Save Checkpoint

In [ ]:
from pathlib import Path
import torch

# Save trained encoder + training history
checkpoint = {
    'encoder_state_dict': encoder.state_dict(),
    'wave_dim': WAVE_DIM,
    'training_history': history,
    'train_steps': TRAIN_STEPS,
    'similarity_targets': {
        'different': SIM_DIFFERENT,
        'modified': SIM_MODIFIED,
        'transformed': SIM_TRANSFORMED,
    },
}

save_path = Path('/kaggle/working/gridtowave_contrastive.pt')
torch.save(checkpoint, save_path)
print(f'✓ Saved checkpoint to {save_path}')
print(f'  Size: {save_path.stat().st_size / 1e6:.1f} MB')

## Cell 10: Evaluate Discrimination Quality

In [ ]:
encoder.eval()

print('Evaluating discrimination quality...')
print('=' * 50)

# Test on 100 fresh batches
n_test = 100
results = {
    'diff_correct': 0,
    'mod_correct': 0,
    'trans_correct': 0,
    'total': 0,
}

with torch.no_grad():
    for _ in tqdm(range(n_test), desc='Testing'):
        batch = generate_training_batch(1, GRID_SIZE)
        
        w_a = encoder.encode(batch['anchors'][0], mode='holistic')
        w_d = encoder.encode(batch['differents'][0], mode='holistic')
        w_m = encoder.encode(batch['modifieds'][0], mode='holistic')
        w_t = encoder.encode(batch['transformeds'][0], mode='holistic')
        
        sim_d = F.cosine_similarity(w_a.unsqueeze(0), w_d.unsqueeze(0)).item()
        sim_m = F.cosine_similarity(w_a.unsqueeze(0), w_m.unsqueeze(0)).item()
        sim_t = F.cosine_similarity(w_a.unsqueeze(0), w_t.unsqueeze(0)).item()
        
        # Check ordering: diff < mod < trans
        if sim_d < sim_m:
            results['diff_correct'] += 1
        if sim_m < sim_t:
            results['mod_correct'] += 1
        if sim_d < sim_t:
            results['trans_correct'] += 1
        results['total'] += 1

print('\nDiscrimination Results:')
print(f'  Different < Modified: {results["diff_correct"]/results["total"]*100:.1f}%')
print(f'  Modified < Transformed: {results["mod_correct"]/results["total"]*100:.1f}%')
print(f'  Different < Transformed: {results["trans_correct"]/results["total"]*100:.1f}%')

# Overall score
score = (results['diff_correct'] + results['mod_correct'] + results['trans_correct']) / (3 * results['total'])
print(f'\n✓ Discrimination score: {score*100:.1f}%')

## Cell 11: Upload to HuggingFace Hub

In [ ]:
# Upload trained checkpoint to HuggingFace
from huggingface_hub import HfApi, login
import os

# Get token from Kaggle secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret('HF_TOKEN')
except:
    hf_token = os.environ.get('HF_TOKEN')

if hf_token:
    login(token=hf_token)
    api = HfApi()
    
    api.upload_file(
        path_or_fileobj=str(save_path),
        path_in_repo='checkpoints/gridtowave_contrastive.pt',
        repo_id='UnseenGAP/FLUX',
        repo_type='model',
    )
    print('✓ Uploaded to HuggingFace Hub')
else:
    print('⚠ HF_TOKEN not found, skipping upload')

## Cell 12: Summary

In [ ]:
print('=' * 60)
print('FLUX GRIDTOWAVE CONTRASTIVE TRAINING COMPLETE')
print('=' * 60)
print()
print(f'Training method: Contrastive learning on random grids')
print(f'Training steps: {TRAIN_STEPS}')
print(f'Final loss: {sum(history["loss"][-10:])/10:.4f}')
print(f'Discrimination score: {score*100:.1f}%')
print()
print('Similarity targets (achieved):')
print(f'  Different: {SIM_DIFFERENT} (actual: {sim_d:.2f})')
print(f'  Modified: {SIM_MODIFIED} (actual: {sim_m:.2f})')
print(f'  Transformed: {SIM_TRANSFORMED} (actual: {sim_t:.2f})')
print()
print(f'Checkpoint: {save_path}')
print('=' * 60)